# Using the DeepSeek API

Author: Manuel Eugenio Morocho Cayamcela, PhD

This short tutorial shows how to call DeepSeek from Python. DeepSeek provides an OpenAI-compatible API, so the same `openai` Python package can be used with a different `base_url`.

> **Useful links:** [DeepSeek API docs](https://api-docs.deepseek.com/) - [DeepSeek models and pricing](https://api-docs.deepseek.com/quick_start/pricing) - [DeepSeek API keys](https://platform.deepseek.com/api_keys) - [OpenAI models](https://platform.openai.com/docs/models) - [OpenAI pricing](https://openai.com/api/pricing/)

Prices change. Always check the official pricing pages before calculating a real budget.

## 1. Setup

Install the OpenAI client. It also works with DeepSeek because the API follows the OpenAI-compatible format.

In [ ]:
%pip install --upgrade openai

## 2. API keys

Create a key at [DeepSeek API keys](https://platform.deepseek.com/api_keys) and another at [OpenAI API keys](https://platform.openai.com/api-keys). Do not publish keys in a notebook or commit them to Git.

For a quick local demonstration, set them as environment variables before starting Jupyter:

```bash
export DEEPSEEK_API_KEY=your-deepseek-key
export OPENAI_API_KEY=your-openai-key
```

In [ ]:
import os
from openai import OpenAI

In [ ]:
# Replace 'your-api-key' with your real API key
os.environ['OPENAI_API_KEY'] = 'your-openai-api-key'  # Replace with your actual OpenAI API key
os.environ['DEEPSEEK_API_KEY'] = 'your-deepseek-api-key'  # Replace with your actual DeepSeek API key

In [ ]:
# Check if the required environment variables are set
if not os.getenv("DEEPSEEK_API_KEY") or not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Set DEEPSEEK_API_KEY and OPENAI_API_KEY before running this cell.")

deepseek_client = OpenAI(
    api_key=os.environ["DEEPSEEK_API_KEY"],
    base_url="https://api.deepseek.com"
)
openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

## 3. A first DeepSeek request

`deepseek-chat` is a general-purpose chat model. `deepseek-reasoner` is intended for more complex reasoning and may be slower or more expensive.

In [ ]:
prompt = "Explain recursion in programming in three simple sentences."

response = deepseek_client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": "You are a clear programming teacher."},
        {"role": "user", "content": prompt}
    ],
    temperature=0.2,
    max_tokens=150
)

print(response.choices[0].message.content)
print("Tokens used:", response.usage.total_tokens)

## 4. Compare DeepSeek and OpenAI

We send the same prompt to both providers and measure elapsed time. Network conditions, server load, prompt length, and model choice affect the result, so one run is not a definitive benchmark.

In [ ]:
import time

def call_model(client, model, prompt):
    start = time.perf_counter()
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        max_tokens=150
    )
    seconds = time.perf_counter() - start
    return response, seconds

comparison_prompt = "Give three practical ways AI can help university students. Use one short sentence per way."

deepseek_response, deepseek_seconds = call_model(deepseek_client, "deepseek-chat", comparison_prompt)
openai_response, openai_seconds = call_model(openai_client, "gpt-4o-mini", comparison_prompt)

print("DeepSeek:", deepseek_seconds, "seconds")
print(deepseek_response.choices[0].message.content)
print("\nOpenAI:", openai_seconds, "seconds")
print(openai_response.choices[0].message.content)

## 5. Estimate the cost

The APIs report input and output tokens. The following example multiplies those tokens by prices per million tokens. **Update the four prices using the official pricing pages before drawing conclusions.** The default values are only classroom examples.

In [ ]:
# Example prices in USD per 1 million tokens; verify current prices first.
deepseek_input_price = 0.28
deepseek_output_price = 0.42
openai_input_price = 0.15
openai_output_price = 0.60

def estimated_cost(response, input_price, output_price):
    input_tokens = response.usage.prompt_tokens
    output_tokens = response.usage.completion_tokens
    cost = (input_tokens * input_price + output_tokens * output_price) / 1_000_000
    return input_tokens, output_tokens, cost

deepseek_usage = estimated_cost(deepseek_response, deepseek_input_price, deepseek_output_price)
openai_usage = estimated_cost(openai_response, openai_input_price, openai_output_price)

print("Provider       Input   Output   Estimated cost (USD)")
print(f"DeepSeek       {deepseek_usage[0]:5d}   {deepseek_usage[1]:6d}   ${deepseek_usage[2]:.8f}")
print(f"OpenAI         {openai_usage[0]:5d}   {openai_usage[1]:6d}   ${openai_usage[2]:.8f}")

## 6. Interpretation

Discuss the result carefully:

- Which response was clearer or more useful?
- Which request was faster in this run?
- Which request was cheaper according to the prices you entered?
- Would the conclusion change with a longer prompt, a longer answer, or another model?

Latency is a measurement from your computer and network. Cost is an estimate based on token usage and the current published prices.

## Individual activity (30 minutes)

1. Change `comparison_prompt` to a question related to your field.
2. Run the comparison at least three times.
3. Record the response time, token counts, estimated cost, and a brief quality judgment for each provider.
4. Write a short conclusion: which model would you choose for your use case, and why?

Do not include API keys in the submitted notebook.

## References

- [DeepSeek API documentation](https://api-docs.deepseek.com/)
- [DeepSeek models and pricing](https://api-docs.deepseek.com/quick_start/pricing)
- [DeepSeek API keys](https://platform.deepseek.com/api_keys)
- [OpenAI API documentation](https://platform.openai.com/docs)
- [OpenAI models](https://platform.openai.com/docs/models)
- [OpenAI pricing](https://openai.com/api/pricing/)